<div style="padding: 1em 0.5em; color: #fff; background-color: #0969da; font-size: 1.2em;">
    Jour 3 — Distillation de connaissances
</div>
<div style="border-left: 2px solid #0969da; min-height: 1.5em;margin-left: 1em;padding: 1em;">
    - Comprendre la distillation : apprendre des probabilités douces du teacher, pas seulement des étiquettes 0/1<br>
    - Aligner les entrées du student (k-mers) sur les fenêtres exactes notées par le teacher<br>
    - Entraîner le student avec la perte hybride et mesurer la précision récupérée<br>
</div>

#### **Votre identité**

Double-cliquez sur cette cellule et complétez, puis exécutez-la (`Maj + Entrée`).

- **Nom & prénom :** _à compléter_
- **Groupe / binôme :** _à compléter_
- **Date :** _à compléter_

<div style="height: 3px; margin: 2em 0 1.5em 0; background: linear-gradient(90deg, #0969da 0%, #0969da 55%, rgba(9,105,218,0.15) 100%);"></div>

Le teacher (embeddings Evo2 + MLP) est précis mais dépend du fait qu'un modèle à 7
milliards de paramètres ait déjà traité chaque entrée. Peut-on compresser ce qu'il a
appris dans quelque chose de minuscule?

**Distillation** : entraîner un petit « student » non seulement sur les étiquettes dures
0/1, mais sur les prédictions *douces* du teacher (sa probabilité, adoucie par une
température) — l'incertitude du teacher porte une information (« *dark knowledge* ») que
les étiquettes dures ne portent pas.

<img src="https://raw.githubusercontent.com/Genereux-akotenou/EEIA-bioAI-Workshop-project/main/day3/assets/distillation_intro.png" />

In [ ]:
import sys
sys.path.append("src")

from pathlib import Path

import torch
from data import load_all
from embeddings import load_supervised_embeddings
from featurize import kmer_matrix
from models.classifier_heads import MLPHead
from models.distillation import train_student, distillation_loss
from eval import evaluate_logits, count_params, measure_latency_torch

EMB_DIR = "../2-data/embeddings"
PROCESSED_DIR = "../2-data/processed"

X_train_emb, y_train, ids_train = load_supervised_embeddings(EMB_DIR, "train")
X_val_emb, y_val, ids_val = load_supervised_embeddings(EMB_DIR, "val")

# on recharge le teacher entraîné au notebook 02 (pas besoin de le ré-entraîner)
TEACHER_PATH = Path("../2-data/models/teacher_mlp.pt")
if not TEACHER_PATH.exists():
    raise FileNotFoundError(
        f"{TEACHER_PATH} introuvable — exécutez d'abord la cellule de sauvegarde "
        "du notebook 02_evo2_embeddings_and_classifier.ipynb."
    )
ckpt = torch.load(TEACHER_PATH)
teacher = MLPHead(d_in=ckpt["d_in"])
teacher.load_state_dict(ckpt["state_dict"])
teacher.eval()  # le teacher ne s'entraîne plus : il ne fait que produire des cibles douces
print("teacher rechargé depuis", TEACHER_PATH)

<div style="height: 3px; margin: 2em 0 1.5em 0; background: linear-gradient(90deg, #0969da 0%, #0969da 55%, rgba(9,105,218,0.15) 100%);"></div>

#### **Faire correspondre les entrées du student aux fenêtres exactes notées par le teacher**

Les embeddings ont été extraits pour un sous-échantillon de fenêtres (voir
`extract_evo2_embeddings.py --max_per_split`), identifiées par `ids`. Il nous faut les
séquences brutes de ces *mêmes* fenêtres pour calculer les caractéristiques k-mer du
student.

In [ ]:
splits = load_all(PROCESSED_DIR)
train_df = splits["train"].set_index("id")
val_df = splits["val"].set_index("id")

# TODO : récupérez les séquences correspondant à ids_train / ids_val (train_df.loc[..., "sequence"])
train_seqs = ...
val_seqs = ...

# TODO : calculez les matrices de k-mer (k=4) pour ces séquences, convertissez en tensors torch
X_train_kmer = ...
X_val_kmer = ...
y_train_t = torch.tensor(y_train, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.float32)
X_train_emb_t = torch.tensor(X_train_emb, dtype=torch.float32)

In [ ]:
# le teacher est déjà entraîné (notebook 02) et gelé : il ne sert qu'à produire les
# cibles douces (soft targets) sur ces mêmes fenêtres
# TODO : calculez les logits du teacher sur X_train_emb_t, sans gradient
with torch.no_grad():
    teacher_logits_train = ...

#### **Entraînons le student avec la perte hybride (entropie croisée + distillation par cibles douces)**

In [ ]:
# TODO : instanciez un student MLPHead volontairement minuscule (d_in=X_train_kmer.shape[1], d_hidden=32)
student = ...

# TODO : entraînez-le avec train_student(...) — 30 époques, temperature=4.0, alpha=0.5
student, history = ...

In [ ]:
# TODO : évaluez le student sur validation, puis calculez metrics/params/latence
student.eval()
with torch.no_grad():
    val_logits = ...
student_metrics = ...
print("distilled student:", student_metrics)
print("params:", ..., "| latency (ms/sample):", ...)

#### **Point de contrôle**

Le student devrait récupérer la majeure partie de la précision du teacher tout en étant
bien plus petit (aucune dépendance à Evo2 à l'inférence — seulement des comptages de
k-mers + un petit MLP) et bien plus rapide. Essayez de faire varier `temperature` et
`alpha` dans `train_student` pour observer le compromis.

Suite : `04_compression_analysis_and_wrapup.ipynb` — mettre tous les modèles sur un seul
graphique.

*Bloqué ? La version complète est dans `solution/03_knowledge_distillation.ipynb`.*

#### **Pour aller plus loin**

**1. Mesurer ce que la distillation apporte vraiment : faites varier `alpha` et `temperature`.**

Les deux hyperparamètres de `train_student(...)` contrôlent d'où vient le signal
d'apprentissage :

- `alpha` dose le mélange entre étiquettes dures et cibles douces. `alpha=1.0` = aucune
  distillation (entraînement supervisé classique), `alpha=0.0` = imitation pure du
  teacher, `alpha=0.5` = moitié-moitié.
- `temperature` contrôle l'aplatissement des probabilités du teacher. À `T=1` ses
  cibles ressemblent déjà à des 0/1 et n'apportent presque rien de plus que les
  étiquettes ; plus `T` monte, plus les écarts de confiance entre fenêtres deviennent
  visibles.

Ré-entraînez le student pour chaque combinaison et relevez l'accuracy de validation :

| | `alpha=1.0` | `alpha=0.5` | `alpha=0.0` |
| --- | --- | --- | --- |
| `T=1` | | | |
| `T=4` | | | |
| `T=10` | | | |

La colonne `alpha=1.0` est votre **témoin** : c'est le même student, entraîné sans le
teacher. L'écart entre cette colonne et les autres *est* le gain de la distillation —
tout le reste n'est que réglage. Attention : chaque case est un entraînement complet,
donc relancez bien un `student = MLPHead(...)` neuf à chaque fois, sinon vous continuez
d'entraîner le précédent.

Questions à se poser : le gain est-il régulier ou y a-t-il un optimum net ? `alpha=0.0`
fait-il mieux ou moins bien que `alpha=0.5` — et qu'est-ce que cela dit sur la fiabilité
du teacher ? Une température trop élevée finit-elle par dégrader le résultat ?

**2. Agrandir le student : capacité ou représentation ?**

Le student est volontairement minuscule (`d_hidden=32`). Que se passe-t-il si on lui
donne plus de paramètres ? Refaites l'entraînement pour
`d_hidden` ∈ {8, 32, 128, 512} et tracez l'accuracy de validation en fonction du nombre
de paramètres (`count_params(student)`).

Vous devriez voir la courbe **monter puis plafonner** bien en dessous du teacher. C'est
le point important : la limite du student n'est pas sa capacité, c'est son **entrée**.
Un histogramme de k-mers a déjà jeté l'ordre des nucléotides et tout le contexte long ;
aucun nombre de neurones ne peut récupérer une information absente des caractéristiques.
L'écart restant avec le teacher est *représentationnel*, pas une question de taille.

Deux choses à surveiller :

- Combinez avec le point 1 : la distillation aide surtout les **petits** students. À
  `d_hidden=512`, `alpha=1.0` et `alpha=0.5` donneront sans doute des résultats proches —
  un gros student n'a plus besoin de l'aide du teacher pour ajuster les étiquettes dures.
- Avec seulement quelques milliers de fenêtres d'entraînement, un `d_hidden=512`
  (~130 000 paramètres) commence à **surapprendre**. Comparez perte d'entraînement et
  perte de validation avant de conclure que « plus grand = moins bon ».

Et pour le Jour 4 : sur le graphique d'efficacité, agrandir le student le déplace vers la
droite (plus de paramètres) sans le faire monter (pas plus de précision) — la pire
direction possible. Le gain de la semaine venait de la *représentation* distillée, pas du
nombre de neurones.

**3. Distiller vers un autre student.**

Rien n'oblige le student à être un MLP sur k-mers : **n'importe quel petit modèle peut
jouer ce rôle**, tant qu'il voit les *mêmes* fenêtres que celles notées par le teacher.
Le CNN one-hot du Jour 1 est un candidat naturel — il apprend ses propres motifs au lieu
de compter des k-mers, et il a donc plus de chances de récupérer ce que le teacher sait.

Si vous voulez essayer :

1. Encodez `train_seqs` / `val_seqs` en one-hot (`one_hot_batch`, longueur 200) plutôt
   qu'en k-mers — ce sont déjà les fenêtres alignées sur `ids_train` / `ids_val`.
2. Instanciez `OneHotCNN(seq_len=200)` comme student.
3. Appelez le même `train_student(...)` avec ces entrées : `teacher_logits_train` ne
   change pas, puisque le teacher note les fenêtres, pas leur représentation.

Questions à se poser : le student CNN récupère-t-il une plus grande part de la précision
du teacher que le student k-mer ? Au prix de combien de paramètres et de combien de
latence supplémentaire ? Et si vous distilliez plutôt vers une **régression logistique**
sur k-mers, jusqu'où peut-on descendre en taille avant que la distillation ne serve plus
à rien ?

<div style="margin-top: 3em;">
  <div style="height: 3px; background: linear-gradient(90deg, #0969da 0%, #0969da 55%, rgba(9,105,218,0.15) 100%);"></div>
  <div style="display: flex; align-items: center; gap: 0.9em; padding: 1.1em 1em; font-size: 0.9em; color: #57606a; background-color: #f2f6fd; border-radius: 0 0 6px 6px;">
    <svg width="34" height="34" viewBox="0 0 34 34" fill="none" style="flex: 0 0 auto;">
      <path d="M9 3c0 7 16 7 16 14S9 24 9 31" stroke="#0969da" stroke-width="2" stroke-linecap="round"/>
      <path d="M25 3c0 7-16 7-16 14s16 7 16 14" stroke="#0969da" stroke-width="2" stroke-linecap="round" opacity="0.45"/>
      <circle cx="17" cy="10" r="1.8" fill="#0969da"/>
      <circle cx="17" cy="24" r="1.8" fill="#0969da"/>
    </svg>
    <div style="flex: 1 1 auto;">
      <div style="color: #0969da; font-weight: 600; letter-spacing: 0.03em;">Fin du Jour 3</div>
      <div>Prochaine &eacute;tape &rarr; <code>day4/04_compression_analysis_and_wrapup.ipynb</code></div>
    </div>
    <div style="flex: 0 0 auto; text-align: right; border-right: 2px solid #0969da; padding-right: 0.9em;">
      <div style="font-weight: 600; color: #24292f;">EEIA &middot; bioAI Workshop</div>
      <div style="font-size: 0.85em;">Semaine 4 &mdash; De l'ADN aux mod&egrave;les compress&eacute;s</div>
    </div>
  </div>
</div>